# 🧠 BRAVE-Net: Full Experiment Notebook
### Burg Residual Augmented Vision Transformer for Parkinson's Disease Detection

**Author:** MD Arif Faysal Nayem | **Supervisor:** Dr. Md. Ohidujjaman  
**Target:** IEEE Access (Q1) · Biomedical Signal Processing and Control (Elsevier, Q1)

---

## 📋 Notebook Contents

| # | Section | Description |
|---|---------|-------------|
| 1 | **Environment Setup** | Install packages, detect GPU, clone repo |
| 2 | **Dataset Setup** | Mount Drive / Kaggle, verify TORGO structure |
| 3 | **Signal Exploration** | Visualise raw PD vs HC audio + spectrograms |
| 4 | **Burg Restoration** | Apply & visualise the physics-informed restoration |
| 5 | **Feature Extraction** | Generate Mel-Spectrogram images for all splits |
| 6 | **Baseline 1** | MFCC + Random Forest (Hassan et al., 2019) |
| 7 | **Baseline 2** | Raw Mel-Spec + ResNet-18 |
| 8 | **Baseline 3** | Raw Mel-Spec + ViT-B/16 (no restoration) |
| 9 | **BRAVE-Net** | Burg-restored Mel-Spec + ViT-B/16 |
| 10 | **Ablation Results** | Full comparison table + ROC curves |
| 11 | **Grad-CAM** | Interpretability heatmaps |
| 12 | **Cross-Dataset** | Generalisation on MDVR-KCL |

---
> ⚠️ **Before running:** Set `DATASET_SOURCE` in Section 2 to either `"gdrive"` or `"kaggle"`

## Section 1 — Environment Setup

In [ ]:
# ── Detect runtime environment ───────────────────────────────────────────────
import sys, os

IN_COLAB  = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")
ENV_NAME  = "Colab" if IN_COLAB else ("Kaggle" if IN_KAGGLE else "Local")

print(f"Runtime: {ENV_NAME}")

# ── Install dependencies ──────────────────────────────────────────────────────
# Skip if already installed (Kaggle pre-installs many packages)
import subprocess

def pip_install(packages: list[str]) -> None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *packages],
        check=True,
    )

REQUIRED = [
    "timm>=0.9.12",
    "grad-cam>=1.4.8",
    "librosa>=0.10.1",
    "soundfile>=0.12.1",
    "PyYAML>=6.0.1",
    "tqdm>=4.66.0",
    "pingouin>=0.5.3",
    "omegaconf>=2.3.0",
    "kagglehub>=0.3.0",
]

pip_install(REQUIRED)
print("✓ Dependencies installed")

In [ ]:
# ── Clone / locate the BRAVE-Net repository ──────────────────────────────────
import os
from pathlib import Path

REPO_URL  = "https://github.com/YOUR_USERNAME/BRAVE-NET.git"   # ← update this
REPO_NAME = "BRAVE-NET"

if IN_COLAB or IN_KAGGLE:
    if not Path(REPO_NAME).exists():
        os.system(f"git clone {REPO_URL}")
    REPO_ROOT = Path(REPO_NAME)
else:
    # Local: assume notebook is inside the repo
    REPO_ROOT = Path("..").resolve()

# Add src to Python path
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
print(f"Working directory: {REPO_ROOT}")

# ── GPU check ─────────────────────────────────────────────────────────────────
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    DEVICE   = "cuda"
    print(f"✓ GPU: {gpu_name}  ({gpu_mem:.1f} GB VRAM)")
elif torch.backends.mps.is_available():
    DEVICE = "mps"
    print("✓ Device: Apple MPS")
else:
    DEVICE = "cpu"
    print("⚠ No GPU found — running on CPU (training will be slow)")

In [ ]:
# ── Standard imports (used throughout) ───────────────────────────────────────
import random
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image
from tqdm.notebook import tqdm
from pathlib import Path

warnings.filterwarnings("ignore")
matplotlib.rcParams.update({
    "figure.dpi": 120,
    "font.size":  11,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(SEED)

print("✓ All imports successful")

## Section 2 — Dataset Setup

Choose your dataset source below, then run the cell.  
- **Google Colab**: upload the TORGO zip to your Drive, or use a shared Drive link  
- **Kaggle**: add the TORGO dataset as a Kaggle Dataset input  
- **Local**: set the path directly

In [ ]:
import kagglehub

# ── Download TORGO from Kaggle Hub (auto-cached after first run) ──────────────
# On Colab: needs a Kaggle API key in ~/.kaggle/kaggle.json (see note below)
# On Kaggle notebooks: add the dataset via Add Data → auto-resolved locally
# Kaggle API key setup on Colab:
#   from google.colab import files; files.upload()   # upload kaggle.json
#   !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
print("Downloading TORGO via kagglehub (cached after first run) ...")
KAGGLE_RAW      = Path(kagglehub.dataset_download("pranaykoppula/torgo-audio"))
print(f"✓ Kaggle dataset at: {KAGGLE_RAW}")

# ── Output directories ─────────────────────────────────────────────────────────
TORGO_RAW_DIR       = REPO_ROOT / "data" / "raw"       / "TORGO"
MDVR_RAW_DIR        = REPO_ROOT / "data" / "raw"       / "MDVR-KCL"
PROCESSED_BRAVE_DIR = REPO_ROOT / "data" / "processed" / "TORGO" / "brave_net"
PROCESSED_RAW_DIR   = REPO_ROOT / "data" / "processed" / "TORGO" / "raw_mel"
MFCC_DIR            = REPO_ROOT / "data" / "processed" / "TORGO" / "mfcc"
RESULTS_DIR         = REPO_ROOT / "results"
FIGURES_DIR         = REPO_ROOT / "figures"
CKPT_DIR            = REPO_ROOT / "checkpoints"

for d in [TORGO_RAW_DIR, MDVR_RAW_DIR, PROCESSED_BRAVE_DIR, PROCESSED_RAW_DIR,
          MFCC_DIR, RESULTS_DIR, FIGURES_DIR, CKPT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("✓ Output directories ready")

In [ ]:
import re, shutil

# ── Speaker ID normalisation ──────────────────────────────────────────────────
# The Kaggle mirror uses FC01/MC01 but the original TORGO tree uses FC1/MC1.
# Build a lookup for all zero-padded control IDs up to 9.
_CTRL_NORM = {f"FC0{i}": f"FC{i}" for i in range(1, 10)}
_CTRL_NORM.update({f"MC0{i}": f"MC{i}" for i in range(1, 10)})

# ── Remap Kaggle flat layout → standard TORGO tree ────────────────────────────
# Kaggle layout : F_Dys/wav_headMic_F01S01/*.wav
# Expected      : F01/Session1/wav_headMic/*.wav
def remap_kaggle_torgo(kaggle_root: Path, dest: Path) -> None:
    GROUP_DIRS = {"F_Dys", "M_Dys", "F_Con", "M_Con"}
    pat        = re.compile(r"wav_(\w+)_([A-Z0-9]+?)S(\d+)$")

    for group_dir in sorted(kaggle_root.iterdir()):
        if group_dir.name not in GROUP_DIRS or not group_dir.is_dir():
            continue
        for subfolder in sorted(group_dir.iterdir()):
            m = pat.match(subfolder.name)
            if not m:
                continue
            mic, raw_spk, sess_num = m.groups()
            speaker      = _CTRL_NORM.get(raw_spk, raw_spk)      # e.g. FC01 → FC1
            session_name = f"Session{int(sess_num)}"              # e.g. S01 → Session1
            target_dir   = dest / speaker / session_name / f"wav_{mic}"
            target_dir.mkdir(parents=True, exist_ok=True)
            for wav in subfolder.glob("*.wav"):
                link = target_dir / wav.name
                if not link.exists():
                    try:
                        link.symlink_to(wav.resolve())
                    except OSError:              # symlinks not supported (some envs)
                        shutil.copy2(wav, link)

# ── Run remap once (idempotent) ───────────────────────────────────────────────
existing_wavs = len(list(TORGO_RAW_DIR.rglob("*.wav")))
if existing_wavs == 0:
    print("Remapping Kaggle folder layout → standard TORGO tree ...")
    remap_kaggle_torgo(KAGGLE_RAW, TORGO_RAW_DIR)
    existing_wavs = len(list(TORGO_RAW_DIR.rglob("*.wav")))

print(f"✓ TORGO at {TORGO_RAW_DIR}  ({existing_wavs:,} .wav files)")

In [ ]:
from src.utils.config import load_config
from src.utils.dataset import scan_torgo_dataset, get_loso_splits

CONFIG = load_config("configs/brave_net_config.yaml")

# Override paths with resolved roots
CONFIG["datasets"]["torgo"]["root"] = str(TORGO_RAW_DIR)
CONFIG["datasets"]["mdvr_kcl"]["root"] = str(MDVR_RAW_DIR)
CONFIG["paths"]["checkpoints"] = str(CKPT_DIR)
CONFIG["paths"]["results"]     = str(RESULTS_DIR)
CONFIG["paths"]["figures"]     = str(FIGURES_DIR)

# Scan dataset
records = scan_torgo_dataset(str(TORGO_RAW_DIR))
splits  = get_loso_splits(records)

pd_records = [r for r in records if r["label"] == 1]
hc_records = [r for r in records if r["label"] == 0]

print(f"Total utterances : {len(records)}")
print(f"  PD (dysarthric): {len(pd_records)}")
print(f"  HC (control)   : {len(hc_records)}")
print(f"LOSO splits      : {len(splits)} speakers")

# Summary table
df_summary = pd.DataFrame(records)
print("\nUtterances per speaker:")
display(df_summary.groupby(["speaker_id", "group"]).size().rename("count").reset_index())

## Section 3 — Signal Exploration (EDA)
Visualise raw waveform, spectrogram, and LP residual for a PD vs HC speaker pair.

In [ ]:
import librosa
import librosa.display
from src.utils.audio_utils import preprocess_audio
from src.features.burg_lp import compute_burg_lpc
from src.features.residual_error import compute_residual_signal

SR = CONFIG["audio"]["sample_rate"]

# Pick one PD and one HC file for visualisation
pd_file = pd_records[0]["audio_path"]  if pd_records else None
hc_file = hc_records[0]["audio_path"]  if hc_records else None

def _load(path):
    s, sr = preprocess_audio(path, sample_rate=SR)
    return s.astype(np.float64), sr

def plot_signal_comparison(pd_path, hc_path):
    pd_sig, _ = _load(pd_path)
    hc_sig, _ = _load(hc_path)

    # Use first 1 s
    pd_sig = pd_sig[:SR]
    hc_sig = hc_sig[:SR]

    lpc_order = CONFIG["burg_lp"]["lpc_order"]
    pd_lpc, _, _ = compute_burg_lpc(pd_sig[:400], order=lpc_order)
    hc_lpc, _, _ = compute_burg_lpc(hc_sig[:400], order=lpc_order)
    pd_res = compute_residual_signal(pd_sig[:400], pd_lpc)
    hc_res = compute_residual_signal(hc_sig[:400], hc_lpc)

    fig, axes = plt.subplots(3, 2, figsize=(14, 10))
    t_full = np.arange(SR) / SR
    t_short = np.arange(400) / SR

    # Row 1: Waveforms
    for ax, sig, title, colour in [
        (axes[0,0], pd_sig, "PD — Waveform", "#E53935"),
        (axes[0,1], hc_sig, "HC — Waveform", "#1E88E5"),
    ]:
        ax.plot(t_full, sig, lw=0.6, color=colour, alpha=0.85)
        ax.set_title(title, fontweight="bold")
        ax.set_xlabel("Time (s)"); ax.set_ylabel("Amplitude")
        ax.grid(alpha=0.25)

    # Row 2: Mel-Spectrograms
    for ax, sig, title in [
        (axes[1,0], pd_sig, "PD — Mel-Spectrogram"),
        (axes[1,1], hc_sig, "HC — Mel-Spectrogram"),
    ]:
        mel = librosa.feature.melspectrogram(y=sig.astype(np.float32), sr=SR, n_mels=128)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        img = librosa.display.specshow(mel_db, sr=SR, x_axis="time", y_axis="mel", ax=ax, cmap="viridis")
        ax.set_title(title, fontweight="bold")
        plt.colorbar(img, ax=ax, format="%+2.0f dB")

    # Row 3: LP Residuals (first 400 samples)
    for ax, res, title, colour in [
        (axes[2,0], pd_res, "PD — LP Residual (first 25ms frame)", "#E53935"),
        (axes[2,1], hc_res, "HC — LP Residual (first 25ms frame)", "#1E88E5"),
    ]:
        ax.plot(t_short, res, lw=0.8, color=colour, alpha=0.85)
        ax.set_title(title, fontweight="bold")
        ax.set_xlabel("Time (s)"); ax.set_ylabel("Residual Amplitude")
        ax.grid(alpha=0.25)
        # Highlight tremor band annotation
        ax.axhline(0, color="black", lw=0.5, alpha=0.3)

    plt.suptitle(
        "Signal Comparison: PD vs HC\n"
        "The LP residual (Row 3) encodes glottal irregularities — "
        "this is the core PD biomarker BRAVE-Net preserves.",
        fontsize=12, fontweight="bold", y=1.01,
    )
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "eda_signal_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()

if pd_file and hc_file:
    plot_signal_comparison(pd_file, hc_file)
else:
    print("⚠ No audio files found — run Section 2 to load dataset first")

## Section 4 — Burg Signal Restoration
Apply the physics-informed restoration (Ohidujjaman et al., 2025) and visualise its effect on a PD utterance.

In [ ]:
from src.features.residual_error import restore_full_signal
from src.evaluation.visualize import plot_restoration_comparison
from src.utils.audio_utils import compute_snr, compute_spectral_flatness_delta

if pd_file:
    pd_raw, _ = _load(pd_file)
    pd_raw = pd_raw.astype(np.float64)[:SR*2]   # 2 seconds

    rest_cfg = CONFIG["restoration"]
    burg_cfg = CONFIG["burg_lp"]

    pd_restored = restore_full_signal(
        pd_raw,
        sample_rate=SR,
        lpc_order=burg_cfg["lpc_order"],
        frame_length_ms=burg_cfg["frame_length_ms"],
        hop_length_ms=burg_cfg["hop_length_ms"],
        tremor_band_hz=tuple(rest_cfg["tremor_band_hz"]),
        glottal_band_hz=tuple(rest_cfg["glottal_band_hz"]),
        amplification_alpha=rest_cfg["amplification_alpha"],
        residual_weight=rest_cfg["residual_weight"],
    )

    snr   = compute_snr(pd_raw, pd_restored)
    sf_d  = compute_spectral_flatness_delta(pd_raw, pd_restored)
    print(f"SNR change         : {snr:+.2f} dB")
    print(f"Spectral flatness Δ: {sf_d:+.4f}  (+ = less noise-like)")

    fig = plot_restoration_comparison(
        pd_raw, pd_restored, SR,
        save_path=FIGURES_DIR / "restoration_comparison.png",
    )
    plt.show()
else:
    print("⚠ No PD audio file available")

## Section 5 — Feature Extraction
Generate Mel-Spectrogram images for all LOSO splits.  
**⚠ This is the slow step** — ~15–30 min on Colab free tier. Results are cached so it runs once.

In [ ]:
from src.features.feature_pipeline import (
    extract_brave_features, extract_mfcc_features, process_dataset_to_images
)

def build_loso_manifests(out_dir: Path, apply_restoration: bool, n_jobs: int = 2) -> None:
    """Build train/test manifest JSON files for every LOSO fold."""
    out_dir.mkdir(parents=True, exist_ok=True)
    mode = "brave_net" if apply_restoration else "raw_mel"

    for split in tqdm(splits, desc=f"Extracting [{mode}] LOSO manifests"):
        test_speaker = split["test_speaker"]
        train_recs   = split["train_records"]
        test_recs    = split["test_records"]

        for subset, recs, tag in [("train", train_recs, "train"), ("test", test_recs, "test")]:
            manifest_path = out_dir / f"{tag}_{test_speaker}_manifest.json"
            if manifest_path.exists():
                continue   # Already processed — skip

            img_dir = out_dir / "images" / f"{tag}_{test_speaker}"
            manifest = process_dataset_to_images(
                audio_paths=[r["audio_path"] for r in recs],
                labels=[r["label"] for r in recs],
                output_dir=img_dir,
                config=CONFIG,
                apply_restoration=apply_restoration,
                n_jobs=n_jobs,
            )
            for i, item in enumerate(manifest):
                item["speaker_id"] = recs[i]["speaker_id"]
            with open(manifest_path, "w") as f:
                json.dump(manifest, f, indent=2)

    print(f"  ✓ [{mode}] manifests ready → {out_dir}")


# ── Burg-restored images (for BRAVE-Net) ─────────────────────────────────────
build_loso_manifests(PROCESSED_BRAVE_DIR, apply_restoration=True, n_jobs=2)

# ── Raw Mel-Spectrogram images (for baselines 2 & 3) ─────────────────────────
build_loso_manifests(PROCESSED_RAW_DIR, apply_restoration=False, n_jobs=2)

In [ ]:
# ── MFCC features for Random Forest baseline ─────────────────────────────────
mfcc_path = MFCC_DIR / "features.npz"

if not mfcc_path.exists():
    print("Extracting MFCC features for RF baseline ...")
    all_features, all_labels, all_speakers = [], [], []
    for rec in tqdm(records, desc="MFCC"):
        feat = extract_mfcc_features(rec["audio_path"], CONFIG)
        all_features.append(feat)
        all_labels.append(rec["label"])
        all_speakers.append(rec["speaker_id"])

    np.savez(
        mfcc_path,
        features=np.array(all_features),
        labels=np.array(all_labels),
        speakers=np.array(all_speakers),
    )
    print(f"  ✓ MFCC features → {mfcc_path}")
else:
    print(f"  ✓ MFCC features already exist at {mfcc_path}")

data_mfcc = np.load(mfcc_path, allow_pickle=True)
print(f"  Shape: {data_mfcc['features'].shape}  (n_samples × feature_dim)")

## Section 6 — Baseline 1: MFCC + Random Forest
Replicates Hassan et al. (2019). Uses 13 MFCCs + delta + delta-delta (78-dim vector) with Leave-One-Speaker-Out CV.

In [ ]:
from sklearn.model_selection import LeaveOneGroupOut
from src.models.baselines import build_rf_classifier
from src.training.metrics import compute_all_metrics, aggregate_loso_results

results_rf_path = RESULTS_DIR / "baseline_rf_loso_results.json"

if results_rf_path.exists():
    with open(results_rf_path) as f:
        rf_cv = json.load(f)
    print("✓ Loaded cached RF results")
else:
    data_mfcc = np.load(mfcc_path, allow_pickle=True)
    X, y, groups = data_mfcc["features"], data_mfcc["labels"].astype(int), data_mfcc["speakers"]

    loso     = LeaveOneGroupOut()
    per_fold = []

    for train_idx, test_idx in tqdm(loso.split(X, y, groups), total=len(np.unique(groups)), desc="RF LOSO"):
        X_tr, X_te = X[train_idx], X[test_idx]
        y_tr, y_te = y[train_idx], y[test_idx]

        clf    = build_rf_classifier(n_estimators=500)
        clf.fit(X_tr, y_tr)
        y_pred = clf.predict(X_te)
        y_prob = clf.predict_proba(X_te)[:, 1]
        metrics = compute_all_metrics(y_te, y_pred, y_prob)
        per_fold.append(metrics)
        speaker = groups[test_idx[0]]
        print(f"  {speaker}: F1={metrics['f1']:.4f}  Sens={metrics['sensitivity']:.4f}  Spec={metrics['specificity']:.4f}")

    aggregate = aggregate_loso_results(per_fold)
    rf_cv = {"per_fold": per_fold, "aggregate": aggregate}
    with open(results_rf_path, "w") as f:
        json.dump(rf_cv, f, indent=2)

agg = rf_cv["aggregate"]
print(f"\n{'─'*45}")
print(f"  Baseline 1 — MFCC + Random Forest")
print(f"{'─'*45}")
for k in ["accuracy", "sensitivity", "specificity", "f1", "auc_roc", "mcc"]:
    v = agg.get(k, {})
    print(f"  {k:<14}: {v.get('mean',0):.4f} ± {v.get('std',0):.4f}")

## Section 7 — Baseline 2: ResNet-18 + Raw Mel-Spectrogram
Fine-tunes a pretrained ResNet-18 on raw (un-restored) mel-spectrogram images using LOSO-CV.
Provides the "visual backbone only — no signal restoration" ablation point.

In [ ]:
import torch
from src.models.baselines import build_baseline
from src.utils.dataset import get_loso_splits, scan_torgo_dataset
from src.training.trainer import run_loso_cv

results_rn18_path = RESULTS_DIR / "baseline_resnet18_loso_results.json"

if results_rn18_path.exists():
    with open(results_rn18_path) as f:
        rn18_cv = json.load(f)
    print("✓ Loaded cached ResNet-18 results")
else:
    # Re-scan to get LOSO split objects (speaker-level)
    records   = scan_torgo_dataset(str(TORGO_RAW_DIR))
    splits    = get_loso_splits(records)

    def rn18_builder():
        return build_baseline("resnet18", CONFIG)

    rn18_cv = run_loso_cv(
        model_builder  = rn18_builder,
        splits         = splits,
        processed_dir  = str(PROCESSED_RAW_DIR),
        config         = CONFIG,
        device         = DEVICE,
        run_name       = "baseline_resnet18",
    )
    with open(results_rn18_path, "w") as f:
        json.dump(rn18_cv, f, indent=2)

agg = rn18_cv["aggregate"]
print(f"\n{'─'*50}")
print(f"  Baseline 2 — ResNet-18 + Raw Mel-Spectrogram")
print(f"{'─'*50}")
for k in ["accuracy", "sensitivity", "specificity", "f1", "auc_roc", "mcc"]:
    v = agg.get(k, {})
    print(f"  {k:<14}: {v.get('mean',0):.4f} ± {v.get('std',0):.4f}")

## Section 8 — Baseline 3: ViT-B/16 + Raw Mel-Spectrogram
Same ViT-B/16 backbone as BRAVE-Net but fed **raw (un-restored)** mel-spectrogram images.
Isolates the contribution of the Burg signal restoration stage.

In [ ]:
results_vit_raw_path = RESULTS_DIR / "baseline_vit_raw_loso_results.json"

if results_vit_raw_path.exists():
    with open(results_vit_raw_path) as f:
        vit_raw_cv = json.load(f)
    print("✓ Loaded cached ViT-raw results")
else:
    records = scan_torgo_dataset(str(TORGO_RAW_DIR))
    splits  = get_loso_splits(records)

    def vit_raw_builder():
        return build_baseline("vit_raw", CONFIG)

    vit_raw_cv = run_loso_cv(
        model_builder  = vit_raw_builder,
        splits         = splits,
        processed_dir  = str(PROCESSED_RAW_DIR),
        config         = CONFIG,
        device         = DEVICE,
        run_name       = "baseline_vit_raw",
    )
    with open(results_vit_raw_path, "w") as f:
        json.dump(vit_raw_cv, f, indent=2)

agg = vit_raw_cv["aggregate"]
print(f"\n{'─'*52}")
print(f"  Baseline 3 — ViT-B/16 + Raw Mel-Spectrogram")
print(f"{'─'*52}")
for k in ["accuracy", "sensitivity", "specificity", "f1", "auc_roc", "mcc"]:
    v = agg.get(k, {})
    print(f"  {k:<14}: {v.get('mean',0):.4f} ± {v.get('std',0):.4f}")

## Section 9 — BRAVE-Net: ViT-B/16 + Burg Signal Restoration (Proposed Method)
The full proposed model.  
**Stage 1** → Burg LP analysis + physics-informed residual restoration  
**Stage 2** → ViT-B/16 fine-tuned on 224×224 viridis mel-spectrogram images  
Trained with LOSO-CV, AdamW + cosine LR, AMP, early-stopping (patience=15).

In [ ]:
from src.models.brave_net import build_model

results_brave_path = RESULTS_DIR / "brave_net_loso_results.json"

if results_brave_path.exists():
    with open(results_brave_path) as f:
        brave_cv = json.load(f)
    print("✓ Loaded cached BRAVE-Net results")
else:
    records = scan_torgo_dataset(str(TORGO_RAW_DIR))
    splits  = get_loso_splits(records)

    def brave_builder():
        return build_model(CONFIG)

    brave_cv = run_loso_cv(
        model_builder  = brave_builder,
        splits         = splits,
        processed_dir  = str(PROCESSED_BRAVE_DIR),
        config         = CONFIG,
        device         = DEVICE,
        run_name       = "brave_net",
    )
    with open(results_brave_path, "w") as f:
        json.dump(brave_cv, f, indent=2)

agg = brave_cv["aggregate"]
print(f"\n{'─'*52}")
print(f"  BRAVE-Net — ViT-B/16 + Burg Restoration")
print(f"{'─'*52}")
for k in ["accuracy", "sensitivity", "specificity", "f1", "auc_roc", "mcc"]:
    v = agg.get(k, {})
    print(f"  {k:<14}: {v.get('mean',0):.4f} ± {v.get('std',0):.4f}")

## Section 10 — Ablation Results: Comparison Table & ROC Curves
Consolidates all four runs into a single publication-ready table.  
Plots LOSO per-metric bar charts and overlay ROC curves.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from src.evaluation.visualize import plot_roc_curves, plot_loso_metric_comparison

# ── 1. Build comparison DataFrame ────────────────────────────────────────────
METRIC_KEYS = ["accuracy", "sensitivity", "specificity", "f1", "auc_roc", "mcc"]

def _fmt(agg, k):
    v = agg.get(k, {})
    return f"{v.get('mean',0):.4f} ± {v.get('std',0):.4f}"

runs = {
    "MFCC + RF"          : rf_cv["aggregate"],
    "ResNet-18 + Raw"    : rn18_cv["aggregate"],
    "ViT + Raw"          : vit_raw_cv["aggregate"],
    "BRAVE-Net (Ours)"   : brave_cv["aggregate"],
}

rows = {name: {k: _fmt(agg, k) for k in METRIC_KEYS} for name, agg in runs.items()}
df   = pd.DataFrame(rows).T
df.index.name = "Model"
print("=" * 80)
print("ABLATION STUDY — TORGO LOSO-CV (mean ± std)")
print("=" * 80)
print(df.to_string())
print("=" * 80)
df.to_csv(RESULTS_DIR / "ablation_table.csv")
print(f"\n✓ Saved → {RESULTS_DIR / 'ablation_table.csv'}")

# ── 2. Bar chart per metric ───────────────────────────────────────────────────
fig = plot_loso_metric_comparison(
    results_dict = {
        name: {k: agg.get(k, {}).get("mean", 0) for k in METRIC_KEYS}
        for name, agg in runs.items()
    },
    save_path = FIGURES_DIR / "ablation_bar_chart.png",
)
plt.tight_layout()
plt.show()
print(f"✓ Saved → {FIGURES_DIR / 'ablation_bar_chart.png'}")

In [ ]:
# ── ROC curves: collect pooled y_true / y_prob per model ─────────────────────
# Each per_fold entry must contain "y_true" and "y_prob" lists.
# run_loso_cv returns those inside each fold dict.

def _pool(cv_result, key):
    """Concatenate per-fold arrays stored under `key`."""
    arrays = [f[key] for f in cv_result.get("per_fold", []) if key in f]
    return np.concatenate(arrays) if arrays else np.array([])

roc_data = {}
for name, cv in [("MFCC + RF",       rf_cv),
                 ("ResNet-18 + Raw",  rn18_cv),
                 ("ViT + Raw",        vit_raw_cv),
                 ("BRAVE-Net (Ours)", brave_cv)]:
    y_true = _pool(cv, "y_true")
    y_prob = _pool(cv, "y_prob")
    if len(y_true) > 0:
        roc_data[name] = {"y_true": y_true.tolist(), "y_prob": y_prob.tolist()}

if roc_data:
    fig = plot_roc_curves(roc_data, save_path=FIGURES_DIR / "roc_curves.png")
    plt.tight_layout()
    plt.show()
    print(f"✓ ROC curves saved → {FIGURES_DIR / 'roc_curves.png'}")
else:
    print("⚠  Per-fold y_true/y_prob not found — re-run training cells first.")

## Section 11 — Grad-CAM Visualisation
Generates class-activation heatmaps for BRAVE-Net predictions on held-out TORGO utterances.  
The ViT reshape transform drops the CLS token and reinterprets the 196 patch tokens as a 14×14 spatial map.  
We expect the highest activation to align with the 100–300 Hz glottal harmonic band.

In [ ]:
import glob
from src.models.brave_net import build_model
from src.utils.dataset import BraveNetDataset, get_vit_transforms
from src.evaluation.visualize import (
    generate_gradcam_heatmap,
    plot_gradcam_overlay,
    get_vit_gradcam_target_layers,
    reshape_transform_vit,
)

# ── Locate best BRAVE-Net checkpoint ─────────────────────────────────────────
ckpt_paths = sorted(glob.glob(str(RESULTS_DIR / "checkpoints" / "brave_net_fold*.pt")))
if not ckpt_paths:
    print("⚠  No checkpoints found. Run Section 9 first, or upload checkpoint(s) to RESULTS_DIR/checkpoints/")
else:
    CKPT = ckpt_paths[-1]           # last fold checkpoint as representative
    print(f"Using checkpoint: {CKPT}")

    brave_model = build_model(CONFIG).to(DEVICE)
    brave_model.load_state_dict(torch.load(CKPT, map_location=DEVICE))
    brave_model.eval()

    # ── Grab a few sample images (PD + HC) ───────────────────────────────────
    brave_imgs  = sorted(glob.glob(str(PROCESSED_BRAVE_DIR / "**" / "*.png"), recursive=True))
    pd_imgs     = [p for p in brave_imgs if "/1/" in p or "_label1" in p][:4]
    hc_imgs     = [p for p in brave_imgs if "/0/" in p or "_label0" in p][:4]
    sample_imgs = pd_imgs + hc_imgs
    labels      = [1] * len(pd_imgs) + [0] * len(hc_imgs)

    if not sample_imgs:
        print("⚠  Could not find sample images. Check PROCESSED_BRAVE_DIR structure.")
    else:
        transform   = get_vit_transforms(image_size=224, augment=False)
        target_layer = get_vit_gradcam_target_layers(brave_model)
        n_show = min(4, len(sample_imgs))
        fig, axes = plt.subplots(2, n_show, figsize=(n_show * 4, 8))

        for i in range(n_show):
            img_path = sample_imgs[i]
            label    = labels[i]
            raw_img  = plt.imread(img_path)

            cam = generate_gradcam_heatmap(
                model            = brave_model,
                img_path         = img_path,
                target_layers    = target_layer,
                reshape_transform = reshape_transform_vit,
                device           = DEVICE,
                transform        = transform,
            )

            axes[0, i].imshow(raw_img)
            axes[0, i].set_title(f"{'PD' if label==1 else 'HC'} — Input", fontsize=9)
            axes[0, i].axis("off")

            plot_gradcam_overlay(raw_img, cam, ax=axes[1, i])
            axes[1, i].set_title("Grad-CAM", fontsize=9)
            axes[1, i].axis("off")

        plt.suptitle("BRAVE-Net Grad-CAM Activations (Top: Input  |  Bottom: Heatmap)",
                     fontsize=11, y=1.01)
        plt.tight_layout()
        out_path = FIGURES_DIR / "gradcam_examples.png"
        plt.savefig(out_path, bbox_inches="tight", dpi=150)
        plt.show()
        print(f"✓ Saved → {out_path}")

## Section 12 — Cross-Dataset Generalisation (MDVR-KCL)
Evaluates the best BRAVE-Net fold checkpoint **without re-training** on MDVR-KCL.  
Tests zero-shot generalisation across recording conditions, microphones, and speaker demographics.

In [ ]:
# ── Step 1: extract MDVR-KCL features ────────────────────────────────────────
from src.utils.dataset import scan_mdvrkc_dataset
from src.features.feature_pipeline import process_dataset_to_images
from src.evaluation.evaluate import evaluate_model
from src.utils.dataset import BraveNetDataset, get_vit_transforms
from torch.utils.data import DataLoader

MDVR_RAW_DIR   = REPO_ROOT / "data" / "raw"   / "MDVR-KCL"
MDVR_PROC_DIR  = REPO_ROOT / "data" / "processed" / "MDVR-KCL"
MDVR_MANIFEST  = MDVR_PROC_DIR / "manifest.csv"
MDVR_PROC_DIR.mkdir(parents=True, exist_ok=True)

if not MDVR_MANIFEST.exists():
    if not MDVR_RAW_DIR.exists() or not any(MDVR_RAW_DIR.iterdir()):
        print("⚠  MDVR-KCL raw audio not found. Mount or upload to:", MDVR_RAW_DIR)
        print("   Skipping cross-dataset evaluation.")
    else:
        mdvr_records = scan_mdvrkc_dataset(str(MDVR_RAW_DIR))
        print(f"Found {len(mdvr_records)} MDVR-KCL records.")
        process_dataset_to_images(
            records          = mdvr_records,
            output_dir       = str(MDVR_PROC_DIR),
            config           = CONFIG,
            apply_restoration = True,
            n_jobs           = 4,
        )
        print(f"✓ MDVR-KCL features saved → {MDVR_PROC_DIR}")
else:
    print(f"✓ MDVR-KCL features already extracted.")

# ── Step 2: run inference with best BRAVE-Net checkpoint ─────────────────────
if MDVR_MANIFEST.exists() and ckpt_paths:
    transform    = get_vit_transforms(image_size=224, augment=False)
    mdvr_dataset = BraveNetDataset(str(MDVR_MANIFEST), transform=transform)
    mdvr_loader  = DataLoader(mdvr_dataset, batch_size=32, shuffle=False, num_workers=2)

    brave_model.eval()
    eval_result = evaluate_model(brave_model, mdvr_loader, DEVICE, return_embeddings=False)
    metrics     = compute_all_metrics(
        np.array(eval_result["y_true"]),
        np.array(eval_result["y_pred"]),
        np.array(eval_result["y_prob"]),
    )

    print(f"\n{'─'*52}")
    print(f"  Cross-Dataset — MDVR-KCL (zero-shot)")
    print(f"{'─'*52}")
    for k in ["accuracy", "sensitivity", "specificity", "f1", "auc_roc", "mcc"]:
        print(f"  {k:<14}: {metrics.get(k, 0):.4f}")

    with open(RESULTS_DIR / "mdvr_kcl_cross_dataset.json", "w") as f:
        json.dump(metrics, f, indent=2)
    print(f"\n✓ Cross-dataset results saved → {RESULTS_DIR / 'mdvr_kcl_cross_dataset.json'}")

## Section 13 — Final Summary & Artefact Export
Prints the consolidated results table, lists all saved artefacts, and optionally zips everything for download.

In [ ]:
import zipfile, os

# ── Final results table ───────────────────────────────────────────────────────
all_results = {}
for name, path in [
    ("MFCC + RF",        RESULTS_DIR / "baseline_rf_loso_results.json"),
    ("ResNet-18 + Raw",  RESULTS_DIR / "baseline_resnet18_loso_results.json"),
    ("ViT + Raw",        RESULTS_DIR / "baseline_vit_raw_loso_results.json"),
    ("BRAVE-Net (Ours)", RESULTS_DIR / "brave_net_loso_results.json"),
]:
    if path.exists():
        with open(path) as f:
            data = json.load(f)
        all_results[name] = data.get("aggregate", {})

if all_results:
    rows = {
        name: {k: f"{agg.get(k,{}).get('mean',0):.4f} ± {agg.get(k,{}).get('std',0):.4f}"
               for k in ["accuracy","sensitivity","specificity","f1","auc_roc","mcc"]}
        for name, agg in all_results.items()
    }
    summary_df = pd.DataFrame(rows).T
    print("\n" + "=" * 80)
    print("  FINAL ABLATION RESULTS — TORGO LOSO-CV")
    print("=" * 80)
    print(summary_df.to_string())
    print("=" * 80)

# ── List saved artefacts ──────────────────────────────────────────────────────
print("\n📁 Saved artefacts:")
for p in sorted(RESULTS_DIR.rglob("*")):
    if p.is_file():
        size = p.stat().st_size
        print(f"  {p.relative_to(REPO_ROOT)}  ({size/1024:.1f} KB)")

# ── Optional: zip results for download ───────────────────────────────────────
ZIP_PATH = REPO_ROOT / "brave_net_results.zip"
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in RESULTS_DIR.rglob("*"):
        if p.is_file():
            zf.write(p, p.relative_to(REPO_ROOT))
    for p in FIGURES_DIR.rglob("*"):
        if p.is_file():
            zf.write(p, p.relative_to(REPO_ROOT))

print(f"\n✅  All done! Results zipped → {ZIP_PATH}")

# ── Colab download helper ─────────────────────────────────────────────────────
if IS_COLAB:
    from google.colab import files
    files.download(str(ZIP_PATH))
elif IS_KAGGLE:
    print(f"Download from Kaggle Output panel: {ZIP_PATH}")